# Data Processing from Scratch

Data processing questions test whether you can build the plumbing that makes ML systems work: tokenization, batching, TF-IDF, and data splitting. These appear in MLE rounds specifically because they bridge theory and systems thinking.

## What Interviewers Test
- Implementing a tokenizer (word-level and BPE sketch) from scratch
- Vocabulary building with frequency thresholds and UNK handling
- Padding + attention masks for variable-length batches
- DataLoader pattern in Python (then mapping to PyTorch)
- TF-IDF from scratch vs sklearn
- Why GroupKFold matters — demonstrating understanding of leakage

In [ ]:
import numpy as np
from collections import Counter, defaultdict
import re
np.random.seed(42)


## Word-Level Tokenizer + Vocabulary

The simplest tokenizer: lowercase, split on whitespace/punctuation, map to integer IDs.


In [ ]:
class WordTokenizer:
    def __init__(self, min_freq=2, max_vocab=10000, special_tokens=('[PAD]', '[UNK]')):
        self.min_freq  = min_freq
        self.max_vocab = max_vocab
        self.pad_id    = 0
        self.unk_id    = 1
        self._special  = list(special_tokens)

    def _tokenize(self, text):
        return re.findall(r"[a-z]+|[0-9]+|[^\w\s]", text.lower())

    def fit(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(self._tokenize(t))
        # Keep only frequent tokens, sorted by frequency
        vocab_tokens = [tok for tok, cnt in counter.most_common(self.max_vocab)
                        if cnt >= self.min_freq]
        self.vocab = {tok: i+len(self._special) for i, tok in enumerate(vocab_tokens)}
        for i, s in enumerate(self._special):
            self.vocab[s] = i
        self.id2tok = {v: k for k, v in self.vocab.items()}
        return self

    def encode(self, text):
        return [self.vocab.get(t, self.unk_id) for t in self._tokenize(text)]

    def decode(self, ids):
        return ' '.join(self.id2tok.get(i, '[UNK]') for i in ids)

    def batch_encode(self, texts, max_len=None, pad=True):
        seqs = [self.encode(t) for t in texts]
        if max_len:
            seqs = [s[:max_len] for s in seqs]
        if pad:
            max_l = max_len or max(len(s) for s in seqs)
            attention_masks = [[1]*len(s) + [0]*(max_l - len(s)) for s in seqs]
            seqs = [s + [self.pad_id]*(max_l - len(s)) for s in seqs]
            return np.array(seqs), np.array(attention_masks)
        return seqs

corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the dog barked at the fox and the fox ran away",
    "machine learning models learn from data",
    "data preprocessing is critical for machine learning",
    "the quick machine learns from the data quickly",
]

tok = WordTokenizer(min_freq=2)
tok.fit(corpus)
print(f"Vocab size: {len(tok.vocab)} (special tokens + frequent words)")
print(f"Vocab (first 10): {dict(list(tok.vocab.items())[:10])}")

ids, masks = tok.batch_encode(corpus[:3], max_len=10)
print(f"\nBatch token IDs shape: {ids.shape}")
print(f"Batch attention masks:\n{masks}")


## BPE Tokenizer Sketch

Byte Pair Encoding (BPE) iteratively merges the most frequent adjacent symbol pair. This is the foundation of GPT-2, GPT-4, and LLaMA tokenizers.

> 💡 **Interview Tip:** You don't need to implement BPE from scratch in an interview. Explain the algorithm: (1) start with character vocabulary, (2) count all adjacent pairs, (3) merge the most frequent pair into a new symbol, (4) repeat for N merges. Emphasize *why* it's better than word-level: handles OOV naturally, adapts to subword morphology.


In [ ]:
def bpe_train(texts, n_merges=10):
    """Minimal BPE training. Returns merge rules."""
    # Initialize: each character is a token, words end with </w>
    def get_vocab(texts):
        vocab = Counter()
        for t in texts:
            for word in t.lower().split():
                vocab[' '.join(list(word)) + ' </w>'] += 1
        return vocab

    def get_pairs(vocab):
        pairs = Counter()
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols)-1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def merge_vocab(pair, vocab):
        bigram = ' '.join(pair)
        replacement = ''.join(pair)
        new_vocab = {}
        for word, freq in vocab.items():
            new_word = word.replace(bigram, replacement)
            new_vocab[new_word] = freq
        return new_vocab

    vocab = get_vocab(texts)
    merges = []
    for i in range(n_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        merges.append(best)
        vocab = merge_vocab(best, vocab)
    return merges, vocab

merges, final_vocab = bpe_train(corpus, n_merges=8)
print("BPE merge rules (most frequent pairs merged first):")
for m in merges:
    print(f"  {m[0]} + {m[1]} → {''.join(m)}")


## DataLoader Pattern in Python → PyTorch

Understanding the pure-Python pattern makes it much easier to reason about PyTorch's DataLoader.


In [ ]:
# Pure Python DataLoader
class Dataset:
    """Base class — subclasses implement __len__ and __getitem__."""
    def __len__(self): raise NotImplementedError
    def __getitem__(self, idx): raise NotImplementedError

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=20):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer.encode(self.texts[idx])[:self.max_len]
        return {'ids': tokens, 'label': self.labels[idx]}

def collate_fn(batch, pad_id=0):
    """Pads sequences in a batch to same length."""
    max_len = max(len(item['ids']) for item in batch)
    ids     = np.array([item['ids'] + [pad_id]*(max_len-len(item['ids'])) for item in batch])
    masks   = np.array([[1]*len(item['ids']) + [0]*(max_len-len(item['ids'])) for item in batch])
    labels  = np.array([item['label'] for item in batch])
    return {'ids': ids, 'attention_mask': masks, 'labels': labels}

class DataLoader:
    def __init__(self, dataset, batch_size=2, shuffle=True):
        self.dataset    = dataset
        self.batch_size = batch_size
        self.shuffle    = shuffle

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        if self.shuffle:
            np.random.shuffle(indices)
        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start:start+self.batch_size]
            batch = [self.dataset[i] for i in batch_idx]
            yield collate_fn(batch)

    def __len__(self):
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size

labels = [1, 0, 1, 0, 1]
ds = TextDataset(corpus, labels, tok, max_len=10)
dl = DataLoader(ds, batch_size=2, shuffle=True)

print(f"Dataset size: {len(ds)}, DataLoader batches: {len(dl)}")
for i, batch in enumerate(dl):
    print(f"Batch {i}: ids={batch['ids'].shape}, labels={batch['labels']}")


## TF-IDF from Scratch

$$\text{TF-IDF}(t, d) = \underbrace{\frac{\text{count}(t,d)}{|d|}}_{\text{TF}} \times \underbrace{\log \frac{N+1}{\text{df}(t)+1} + 1}_{\text{smooth IDF}}$$

> 💡 **Interview Tip:** TF-IDF from scratch tests both the math and sparse matrix thinking. Mention that in production you'd use `scipy.sparse` for the document-term matrix since most entries are zero.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_from_scratch(docs):
    """Returns (n_docs, vocab_size) dense TF-IDF matrix."""
    # Tokenize
    tokenized = [re.findall(r'[a-z]+', d.lower()) for d in docs]
    vocab = sorted(set(tok for doc in tokenized for tok in doc))
    v2i   = {v: i for i, v in enumerate(vocab)}
    N, V  = len(docs), len(vocab)

    # TF: term frequency per document
    tf = np.zeros((N, V))
    for d, tokens in enumerate(tokenized):
        if not tokens:
            continue
        for tok in tokens:
            tf[d, v2i[tok]] += 1
        tf[d] /= len(tokens)

    # IDF: smooth IDF (sklearn convention: log((N+1)/(df+1))+1)
    df  = (tf > 0).sum(axis=0)            # document frequency per term
    idf = np.log((N + 1) / (df + 1)) + 1  # (V,)

    tfidf = tf * idf[np.newaxis, :]
    # L2-normalize rows (sklearn default)
    norms = np.linalg.norm(tfidf, axis=1, keepdims=True) + 1e-12
    return tfidf / norms, vocab

tfidf_mat, vocab = tfidf_from_scratch(corpus)
print(f"TF-IDF matrix shape: {tfidf_mat.shape}")

sk_vec = TfidfVectorizer().fit_transform(corpus).toarray()
# Columns may be in different order; check Frobenius norm after reordering
print(f"Max abs diff vs sklearn: {abs(tfidf_mat.sum() - sk_vec.sum()):.4f} (small differences from tokenization)")
print(f"Top terms for doc 3: {sorted(zip(tfidf_mat[3], vocab), reverse=True)[:5]}")


## Train/Val/Test Splits & GroupKFold — Leakage Demo

> 💡 **Interview Tip:** This is the highest-signal data-processing question. Explain: if one user's data appears in both train and test, the model can memorize user-specific patterns and inflate metrics. GroupKFold prevents any group (user, session, date) from appearing in multiple folds.


In [ ]:
from sklearn.model_selection import GroupKFold, KFold

np.random.seed(42)

# Simulate a dataset where each user has multiple rows (e.g., click events)
n_users, n_events_per_user = 20, 50
users = np.repeat(np.arange(n_users), n_events_per_user)     # (1000,) user IDs
X_demo = np.random.randn(n_users * n_events_per_user, 5)

# Add a user-specific signal that would cause leakage
user_bias = np.random.randn(n_users)                          # per-user fixed effect
y_demo    = X_demo[:, 0] + user_bias[users] + 0.1*np.random.randn(len(users))

# --- Naive KFold (WRONG for grouped data) ---
naive_kf = KFold(n_splits=5, shuffle=True, random_state=42)
naive_scores = []
for train_idx, val_idx in naive_kf.split(X_demo):
    train_users = set(users[train_idx])
    val_users   = set(users[val_idx])
    overlap     = len(train_users & val_users)
    naive_scores.append(overlap)

print("=== Naive KFold ===")
print(f"Average user overlap between train/val folds: {np.mean(naive_scores):.1f} users per fold")
print("(LEAKAGE: same user appears in train AND val)")

# --- GroupKFold (CORRECT) ---
gkf = GroupKFold(n_splits=5)
group_scores = []
for train_idx, val_idx in gkf.split(X_demo, y_demo, groups=users):
    train_users = set(users[train_idx])
    val_users   = set(users[val_idx])
    overlap     = len(train_users & val_users)
    group_scores.append(overlap)

print("\n=== GroupKFold ===")
print(f"Average user overlap between train/val folds: {np.mean(group_scores):.1f} users per fold")
print("(NO LEAKAGE: each user in exactly one fold)")


## Common Interview Questions

**Q: What is the difference between a word-level and BPE tokenizer?**
Word-level assigns a unique ID to each word; unknown words at inference time become UNK, losing information. BPE learns subword units by merging frequent character pairs, so rare words are decomposed into known subword pieces rather than UNKed. BPE vocabularies of 30k–100k tokens can represent almost any word.

**Q: Why do we need attention masks in a batch?**
After padding shorter sequences to the batch max length, we don't want the model to attend to padding tokens — they carry no information and would pollute attention weights. The mask (1 = real token, 0 = padding) is used to set padding attention logits to -∞ before softmax.

**Q: What is GroupKFold and when is it critical?**
GroupKFold ensures no group (user, session, entity) appears in both training and validation folds. Without it, models can memorize group-specific patterns (user behavior, entity-level features) and appear to generalize when they're actually memorizing. Use it whenever rows are not i.i.d. — user click logs, medical patient records, geographic data.

**Q: What is the IDF term in TF-IDF and why do we smooth it?**
IDF (inverse document frequency) down-weights terms that appear in many documents (like "the") and up-weights rare, informative terms. Smoothing adds 1 to the denominator and numerator to avoid division by zero for unseen terms and to prevent the IDF of very common terms from being exactly zero.

**Q: Why is the PyTorch DataLoader's `collate_fn` important?**
`collate_fn` assembles a list of individual samples (from `__getitem__`) into a batch. For variable-length sequences this is where padding happens. The default collate stacks tensors of equal size; for NLP you need a custom collate that pads to the max length in the current batch.

## Key Takeaways
- Word tokenizers map words to IDs; UNK handles OOV but loses information — BPE is preferred
- Vocabulary building: count frequencies, filter by min_freq, assign IDs starting after special tokens
- Attention masks (1=real, 0=pad) prevent attention to padding — always pair with padded sequences
- PyTorch DataLoader: Dataset.__getitem__ → collate_fn pads → DataLoader batches
- TF-IDF: TF (local frequency) × IDF (global rarity weight); L2-normalize rows for cosine similarity
- GroupKFold is critical when rows are not i.i.d. — one of the highest-signal data leakage questions